In [15]:
import pandas as pd
import numpy as np
df=pd.read_csv('D:/FM/candle_maker/artifacts/candle_data_15min.csv')
df['date']="06-05-2026"

In [16]:
df['vol_filter'] = (
    df.groupby(['symbol', 'date'])['volume']
    .shift(1)
    .pipe(lambda prev: (df['volume'] > prev).astype(int))
)

In [17]:
df['domination'] = 'sell'
df.loc[df['buy_pct'] > df['sell_pct'], 'domination'] = 'buy'

In [18]:
df['buy_vol_filter'] = (
    df.groupby(['symbol', 'date'])['buy_pct']
    .shift(1)
    .pipe(lambda prev: (df['buy_pct'] > prev).astype(int))
)

In [19]:
df['sell_vol_filter'] = (
    df.groupby(['symbol', 'date'])['sell_pct']
    .shift(1)
    .pipe(lambda prev: (df['sell_pct'] > prev).astype(int))
)

In [20]:
df['candle_domination'] = np.where(
    df['close'] > (df['high'] + df['low']) / 2,
    'buy',
    'sell'
)

In [21]:
df['signal_2'] = 0

df.loc[
    (df['vol_filter'] == 1) & (df['buy_vol_filter'] == 1) & (df['domination'] == 'buy') &(df['candle_domination'] == 'buy'),
    'signal'
] = 1

df.loc[
    (df['vol_filter'] == 1) & (df['sell_vol_filter'] == 1) & (df['domination'] == 'sell') &(df['candle_domination'] == 'sell'),
    'signal'
] = -1

df_filtered = df[df['signal_2'] != 0]

In [22]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_CSV  = "candle_data.csv"       # change to your input file path
OUTPUT_CSV = "candle_data_with_trades_0.3.csv"
CAPITAL    = 100_000                  # 1 lakh
TARGET_PCT = 0.003                # 0.3%


# ─────────────────────────────────────────
# LOAD & SORT
# ─────────────────────────────────────────

df = df.sort_values(['symbol', 'date', 'candle_open']).reset_index(drop=True)


# ─────────────────────────────────────────
# BACKTEST
# ─────────────────────────────────────────
results = []

for idx in df[df['signal'] != 0].index:
    row    = df.loc[idx]
    entry  = row['close']
    signal = row['signal']
    sym    = row['symbol']

    if signal == 1:           # BUY
        sl  = row['low']
        tg  = round(entry * (1 + TARGET_PCT), 2)
    else:                     # SELL
        sl  = row['high']
        tg  = round(entry * (1 - TARGET_PCT), 2)

    qty = int(CAPITAL / entry)

    # Scan future candles of the same symbol for SL / TG
    future     = df[(df['symbol'] == sym) & (df.index > idx)]
    exit_price = None
    exit_crit  = 'OPEN'

    for _, frow in future.iterrows():
        if signal == 1:
            if frow['low'] <= sl:
                exit_price, exit_crit = sl,  'SL';  break
            elif frow['high'] >= tg:
                exit_price, exit_crit = tg,  'TG';  break
        else:
            if frow['high'] >= sl:
                exit_price, exit_crit = sl,  'SL';  break
            elif frow['low'] <= tg:
                exit_price, exit_crit = tg,  'TG';  break

    # If neither hit, exit at last available close for that symbol
    if exit_price is None:
        exit_price = df[df['symbol'] == sym].iloc[-1]['close']
        exit_crit  = 'OPEN'

    pnl = round((exit_price - entry) * qty, 2) if signal == 1 \
          else round((entry - exit_price) * qty, 2)

    trade_result = 'profit' if pnl > 0 else ('loss' if pnl < 0 else 'breakeven')

    results.append({
        'idx':          idx,
        'entry':        round(entry,      2),
        'exit':         round(exit_price, 2),
        'sl':           round(sl,         2),
        'tg':           tg,
        'exit_criteria': exit_crit,
        'trade_result': trade_result,
        'pnl':          pnl,
    })

res_df = pd.DataFrame(results).set_index('idx')

for col in ['entry', 'exit', 'sl', 'tg', 'exit_criteria', 'trade_result', 'pnl']:
    df[col] = res_df[col]


# ─────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────
df_fin=df[['symbol','date','candle_open','signal','entry','exit','sl','tg','exit_criteria','trade_result','pnl']]
df_fin=df_fin[df_fin['signal']!=0]
df_fin.to_csv(OUTPUT_CSV, index=False)


# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
trades = df[df['signal'] != 0]

print("=" * 45)
print("           BACKTEST SUMMARY")
print("=" * 45)
print(f"  Capital        : ₹{CAPITAL:,.0f}")
print(f"  Target         : {TARGET_PCT*100}%")
print(f"  Total trades   : {len(trades)}")
print(f"  Profit (TG hit): {(trades['trade_result'] == 'profit').sum()}")
print(f"  Loss   (SL hit): {(trades['trade_result'] == 'loss').sum()}")
print(f"  Open trades    : {(trades['exit_criteria'] == 'OPEN').sum()}")
print(f"  Net PnL        : ₹{trades['pnl'].sum():,.2f}")
print("=" * 45)
print(f"\nOutput saved to: {OUTPUT_CSV}")

           BACKTEST SUMMARY
  Capital        : ₹100,000
  Target         : 0.3%
  Total trades   : 25
  Profit (TG hit): 7
  Loss   (SL hit): 15
  Open trades    : 6
  Net PnL        : ₹-914.28

Output saved to: candle_data_with_trades_0.3.csv
